# 04 – Urban Heat Hotspot Mapping

## Urban Heat Island Analysis – Voi Town, Kenya

This notebook identifies and visualises relative land surface temperature zones across Voi Town using Landsat 8 and 9 imagery.

The analysis focuses on the 2025 June–September seasonal composite.

In [1]:
import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

ee.Initialize(
    project='voi-urban-heat-island-analysis'
)

print("Earth Engine initialized successfully!")

Earth Engine initialized successfully!


In [2]:
# Define Voi study area
voi_point = ee.Geometry.Point([38.56, -3.39])
voi_aoi = voi_point.buffer(5000)

# Display study area
Map = geemap.Map()
Map.centerObject(voi_aoi, 12)

Map.addLayer(
    voi_aoi,
    {'color': 'red'},
    'Voi Study Area'
)

Map

Map(center=[-3.38999487055625, 38.56000014249946], controls=(WidgetControl(options=['position', 'transparent_b…

In [3]:
# Load Landsat 8
landsat_8 = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterBounds(voi_aoi)
    .filterDate('2016-01-01', '2026-10-01')
)

# Load Landsat 9
landsat_9 = (
    ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
    .filterBounds(voi_aoi)
    .filterDate('2022-01-01', '2026-10-01')
)

# Merge and filter
landsat = (
    landsat_8.merge(landsat_9)
    .filter(ee.Filter.eq('PROCESSING_LEVEL', 'L2SP'))
)

print("Available scenes:", landsat.size().getInfo())

Available scenes: 331


In [5]:
def preprocess_landsat(image):

    # Cloud and shadow masking
    qa = image.select('QA_PIXEL')

    mask = (
        qa.bitwiseAnd(1 << 0).eq(0)
        .And(qa.bitwiseAnd(1 << 1).eq(0))
        .And(qa.bitwiseAnd(1 << 2).eq(0))
        .And(qa.bitwiseAnd(1 << 3).eq(0))
        .And(qa.bitwiseAnd(1 << 4).eq(0))
        .And(qa.bitwiseAnd(1 << 5).eq(0))
    )

    # Remove saturated pixels
    mask = mask.And(image.select('QA_RADSAT').eq(0))

    # Scale optical bands
    optical = (
        image.select([
            'SR_B2', 'SR_B3', 'SR_B4',
            'SR_B5', 'SR_B6', 'SR_B7'
        ])
        .multiply(0.0000275)
        .add(-0.2)
    )

    # Calculate NDVI and NDBI
    ndvi = optical.normalizedDifference(
        ['SR_B5', 'SR_B4']
    ).rename('NDVI')

    ndbi = optical.normalizedDifference(
        ['SR_B6', 'SR_B5']
    ).rename('NDBI')

    # Calculate LST in Celsius
    lst = (
        image.select('ST_B10')
        .multiply(0.00341802)
        .add(149.0)
        .subtract(273.15)
        .rename('LST_C')
    )

    return (
        image.addBands(optical, overwrite=True)
        .addBands([ndvi, ndbi, lst])
        .updateMask(mask)
        .copyProperties(image, ['system:time_start'])
    )

print("Preprocessing function defined successfully!")

Preprocessing function defined successfully!


In [7]:
landsat_processed = landsat.map(preprocess_landsat)

image_2025 = (
    landsat_processed
    .filterDate('2025-06-01', '2025-10-01')
    .median()
    .clip(voi_aoi)
)

print("2025 composite created successfully!")

2025 composite created successfully!


In [8]:
# Calculate LST percentiles
lst_percentiles = image_2025.select('LST_C').reduceRegion(
    reducer=ee.Reducer.percentile([25, 50, 75]),
    geometry=voi_aoi,
    scale=30,
    maxPixels=1e9
).getInfo()

print("2025 LST Percentiles:")
print(lst_percentiles)

2025 LST Percentiles:
{'LST_C_p25': 39.186665310146864, 'LST_C_p50': 40.937877130675126, 'LST_C_p75': 42.686481472261136}


In [9]:
# Extract percentile thresholds
p25 = lst_percentiles['LST_C_p25']
p50 = lst_percentiles['LST_C_p50']
p75 = lst_percentiles['LST_C_p75']

# Classify LST into four heat zones
heat_zones = (
    image_2025.select('LST_C')
    .expression(
        """
        (lst <= p25) ? 1 :
        (lst <= p50) ? 2 :
        (lst <= p75) ? 3 : 4
        """,
        {
            'lst': image_2025.select('LST_C'),
            'p25': p25,
            'p50': p50,
            'p75': p75
        }
    )
    .rename('Heat_Class')
)

print("Four heat zones classified successfully!")

Four heat zones classified successfully!


In [10]:
Map = geemap.Map()

Map.centerObject(voi_aoi, 12)

# Heat classification colours
heat_palette = [
    '2166ac',  # Coolest - Blue
    'fee08b',  # Moderate-cool - Yellow
    'fdae61',  # Hot - Orange
    'd73027'   # Hottest - Red
]

Map.addLayer(
    heat_zones,
    {
        'min': 1,
        'max': 4,
        'palette': heat_palette
    },
    'Voi Urban Heat Zones 2025'
)

Map.addLayer(
    voi_aoi,
    {'color': 'white'},
    'Study Area Boundary'
)

Map

Map(center=[-3.38999487055625, 38.56000014249946], controls=(WidgetControl(options=['position', 'transparent_b…

In [11]:
Map_LST = geemap.Map()

Map_LST.centerObject(voi_aoi, 12)

Map_LST.addLayer(
    image_2025.select('LST_C'),
    {
        'min': 25,
        'max': 50,
        'palette': [
            '313695',
            '74add1',
            'ffffbf',
            'fdae61',
            'a50026'
        ]
    },
    'Land Surface Temperature 2025'
)

Map_LST.addLayer(
    voi_aoi,
    {'color': 'white'},
    'Voi Boundary'
)

Map_LST

Map(center=[-3.38999487055625, 38.56000014249946], controls=(WidgetControl(options=['position', 'transparent_b…

In [12]:
task = ee.batch.Export.image.toDrive(
    image=heat_zones,
    description='Voi_Heat_Zones_2025',
    folder='Voi_UHI_Outputs',
    fileNamePrefix='Voi_Heat_Zones_2025',
    region=voi_aoi,
    scale=30,
    maxPixels=1e13
)

task.start()

print("Heat-zone export task submitted!")

Heat-zone export task submitted!


In [13]:
lst_task = ee.batch.Export.image.toDrive(
    image=image_2025.select('LST_C'),
    description='Voi_LST_2025',
    folder='Voi_UHI_Outputs',
    fileNamePrefix='Voi_LST_2025',
    region=voi_aoi,
    scale=30,
    maxPixels=1e13
)

lst_task.start()

print("LST GeoTIFF export submitted!")

LST GeoTIFF export submitted!


In [15]:
# Export NDVI
ndvi_task = ee.batch.Export.image.toDrive(
    image=image_2025.select('NDVI'),
    description='Voi_NDVI_2025',
    folder='Voi_UHI_Outputs',
    fileNamePrefix='Voi_NDVI_2025',
    region=voi_aoi,
    scale=30,
    maxPixels=1e13
)

ndvi_task.start()


# Export NDBI
ndbi_task = ee.batch.Export.image.toDrive(
    image=image_2025.select('NDBI'),
    description='Voi_NDBI_2025',
    folder='Voi_UHI_Outputs',
    fileNamePrefix='Voi_NDBI_2025',
    region=voi_aoi,
    scale=30,
    maxPixels=1e13
)

ndbi_task.start()

print("NDVI and NDBI export tasks submitted!")

NDVI and NDBI export tasks submitted!
